# Evolution of Vision — CIFAR-10 Architecture Showdown
## Kaggle runner (9-hour unattended mode)

Runs the **entire** training queue: LR sweep → E1 → E5/E6/E7 evaluation → E2 → E4 → upgrades.
Six architectures (MLP, Simple CNN, ResNet-18, ViT, MLP-Mixer, ConvNeXt-mini), all seven
experiments, trimmed seeds and epochs per spec Section 12.

---

### ⚙️ Required Kaggle settings — check these BEFORE you start

| Setting | Value | Why |
|---|---|---|
| **Accelerator** | **GPU T4 × 2** | The queue is split across both GPUs. On 1× T4 the core does **not** fit in 8 h. |
| **Internet** | **ON** | Needed for `pip install`, the CIFAR-10 download, and the E4 pretrained weights. |
| **Environment** | **Always use latest environment** | A notebook pinned to its original environment can get a 2023 image (Python 3.10, PyTorch 2.0). The code still runs there, but the latest environment has a current PyTorch. |
| **Persistence** | Files only (optional) | Not required; everything lands in `/kaggle/working`. |

### ▶️ How to start it

Use **Save Version → Save & Run All (Commit)**, *not* the interactive "Run All".
A committed run keeps going with your browser closed; an interactive session dies when you disconnect.

**Check Kaggle's current per-session GPU limit before you start** (historically ~12 h, and there is a
weekly GPU quota of ~30 h). The default `BUDGET_HOURS = 8` is chosen to sit comfortably under it.

### ⏱️ Expected timeline

Roughly **6.5–8 h** on 2× T4. The planner prints a projected finish time before the long run begins —
read it, and if it exceeds your remaining session time, lower `BUDGET_HOURS`.

In [ ]:
# =============================================================================
# CONFIG -- the only cell you should normally need to edit.
# =============================================================================
from pathlib import Path

# How the repo gets onto this machine.
#   "git"     -> clone from REPO_URL (the repo is public, so no credentials)
#   "dataset" -> you uploaded the repo as a Kaggle Dataset instead
SOURCE = "git"

REPO_URL = "https://github.com/logn1602/cifar10-architecture-showdown.git"
DATASET_SLUG = ""          # only used when SOURCE="dataset"; "" auto-detects

BUDGET_HOURS = 8.0         # spec 12.4 default: ~1 h of slack inside a 9 h window
MODE = "9h"

WORK_DIR = Path("/kaggle/working/cifar10-architecture-showdown")
SPEC_FILE = "CIFAR10_ARCHITECTURE_SHOWDOWN.md"   # the marker that identifies the repo root

# -----------------------------------------------------------------------------
# Shared helper. Every long-running step goes through this rather than a `!`
# magic, because a failing `!` cell does NOT stop a committed run -- it prints
# the error and the notebook sails on to the next cell. That is exactly how you
# come back after 8 hours to an empty results/ directory.
# -----------------------------------------------------------------------------
import subprocess, sys, os, time

def run(cmd, cwd=None, check=True, tee=None):
    """Run a command, streaming output live, optionally teeing to a file."""
    cwd = str(cwd or WORK_DIR)
    print(f"\n$ {' '.join(str(c) for c in cmd)}\n  (cwd={cwd})", flush=True)
    t0 = time.time()
    sink = open(tee, "a", encoding="utf-8") if tee else None
    proc = subprocess.Popen([str(c) for c in cmd], cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, bufsize=1, text=True,
                            encoding="utf-8", errors="replace")
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
            if sink:
                sink.write(line)
    finally:
        proc.wait()
        if sink:
            sink.close()
    dt = time.time() - t0
    print(f"\n[exit {proc.returncode} after {dt/60:.1f} min]", flush=True)
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {cmd}")
    return proc.returncode

PY = sys.executable
print("config loaded")

In [ ]:
# =============================================================================
# 1. Environment probe -- fail fast and loudly if the accelerator is wrong.
# =============================================================================
import shutil, subprocess, sys, os

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch
print(f"python            : {sys.version.split()[0]}")
print(f"torch             : {torch.__version__}  (CUDA build: {torch.version.cuda})")
print(f"cuda available    : {torch.cuda.is_available()}")
n_gpu = torch.cuda.device_count()
print(f"device count      : {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory/1024**3:.1f} GB  sm_{p.major}{p.minor}")
if n_gpu:
    print(f"bf16 supported    : {torch.cuda.is_bf16_supported()}")
print(f"cpu count         : {os.cpu_count()}")
try:
    import psutil
    print(f"RAM               : {psutil.virtual_memory().total/1024**3:.1f} GB")
except ImportError:
    pass
free = shutil.disk_usage("/kaggle/working").free / 1024**3
print(f"/kaggle/working   : {free:.1f} GB free")

assert torch.cuda.is_available(), (
    "No GPU! Settings -> Accelerator -> GPU T4 x2, then restart the session."
)
if n_gpu < 2:
    print("\n" + "!" * 70)
    print(f"!! Only {n_gpu} GPU allocated, expected 2.")
    print("!! The queue still runs, but on 1x T4 the core does NOT fit in 8 h.")
    print("!! Set Accelerator -> GPU T4 x2 and restart, or lower the scope.")
    print("!" * 70)
if free < 15:
    print(f"\n!! Only {free:.1f} GB free -- checkpoints + CIFAR-10 need ~10 GB. Watch for ENOSPC.")

# Kaggle can pin a notebook to the environment it was created in. A pinned
# 2023 image (Python 3.10, PyTorch 2.0) still runs this project -- the code is
# kept compatible with PyTorch 2.0 -- but the latest environment is better.
torch_version = tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2])
if torch_version < (2, 1):
    print("\n" + "!" * 70)
    print(f"!! PyTorch {torch.__version__} on Python {sys.version.split()[0]}: this notebook")
    print("!! is probably PINNED to an old Kaggle environment. The run still works,")
    print("!! but Settings -> Environment -> 'Always use latest environment' is better.")
    print("!" * 70)

In [ ]:
# =============================================================================
# 2. Fetch the repo into /kaggle/working.
#
# Copy, never symlink: /kaggle/input is READ-ONLY and the code writes results,
# checkpoints and the data/ directory.
# =============================================================================
import shutil
from pathlib import Path

def find_dataset_root():
    """Locate the uploaded repo under /kaggle/input by its spec file."""
    if DATASET_SLUG:
        cand = Path("/kaggle/input") / DATASET_SLUG
        hits = [cand] if (cand / SPEC_FILE).exists() else list(cand.rglob(SPEC_FILE))
        if hits:
            return hits[0] if hits[0].is_dir() else hits[0].parent
        raise FileNotFoundError(f"{SPEC_FILE} not found under {cand}")
    hits = sorted(Path("/kaggle/input").glob(f"*/**/{SPEC_FILE}"))
    hits += sorted(Path("/kaggle/input").glob(f"*/{SPEC_FILE}"))
    roots = sorted({h.parent for h in hits})
    if not roots:
        raise FileNotFoundError(
            "No dataset under /kaggle/input contains " + SPEC_FILE + ".\n"
            "Add your repo as a Dataset (see docs/KAGGLE.md), or set SOURCE='git'."
        )
    if len(roots) > 1:
        raise RuntimeError(f"Ambiguous: {SPEC_FILE} found in {roots}. Set DATASET_SLUG.")
    return roots[0]

if (WORK_DIR / SPEC_FILE).exists():
    print(f"repo already present at {WORK_DIR} -- leaving it alone (cell is idempotent)")
elif SOURCE == "dataset":
    src = find_dataset_root()
    print(f"copying {src} -> {WORK_DIR}")
    shutil.copytree(src, WORK_DIR, dirs_exist_ok=True)
elif SOURCE == "git":
    WORK_DIR.parent.mkdir(parents=True, exist_ok=True)
    run(["git", "clone", "--depth", "1", REPO_URL, str(WORK_DIR)], cwd="/kaggle/working")
else:
    raise ValueError(f"SOURCE must be 'dataset' or 'git', got {SOURCE!r}")

os.chdir(WORK_DIR)
assert (WORK_DIR / SPEC_FILE).exists(), f"{SPEC_FILE} missing -- wrong source?"
print(f"\ncwd = {os.getcwd()}\n")

for p in sorted(WORK_DIR.iterdir()):
    print(f"  {p.name}{'/' if p.is_dir() else ''}")
    if p.is_dir() and p.name in {"src", "scripts", "configs"}:
        for q in sorted(p.iterdir()):
            print(f"      {q.name}{'/' if q.is_dir() else ''}")

In [ ]:
# =============================================================================
# 3. Install the extras.
#
# NEVER `pip install torch` here. The Kaggle image already ships a CUDA build;
# a stray torch install can pull a CPU wheel and silently kill GPU training --
# you would not notice until the whole queue ran 100x too slow.
# =============================================================================
import os
os.chdir(WORK_DIR)

before = None
import torch
before = (torch.__version__, torch.version.cuda)

run([PY, "-m", "pip", "install", "-q",
     "timm", "grad-cam", "umap-learn", "pyyaml", "tqdm", "seaborn"], check=False)

# Re-verify in a FRESH interpreter: this process already has torch imported, so
# a downgraded wheel on disk would not show up here.
check = subprocess.run(
    [PY, "-c",
     "import torch;print(torch.__version__, torch.version.cuda, torch.cuda.is_available())"],
    capture_output=True, text=True)
print("after install:", check.stdout.strip() or check.stderr.strip())
assert "True" in check.stdout, (
    f"torch lost CUDA after the pip install (was {before}). "
    "Something pulled a CPU wheel -- restart the session and remove the offending package."
)
print("\nextras installed, torch CUDA intact")

In [ ]:
# =============================================================================
# 4. Pre-flight: what exists, and does the data pipeline reproduce?
#
# The split sha256 printed here MUST match the one from your local
# `python scripts/check_data.py`. data/ is gitignored, so the 45k/5k split is
# regenerated here rather than shipped -- these hashes are the proof that
# "val accuracy" means the same thing on both machines.
# =============================================================================
import os
os.chdir(WORK_DIR)

REQUIRED = {
    "scripts/env_check.py":        "Phase 0",
    "scripts/check_data.py":       "Phase 1",
    "scripts/model_checks.py":     "Phase 2",
    "src/data.py":                 "Phase 1",
    "scripts/plan_budget.py":      "Phase 3",
    "scripts/run_all.py":          "Phase 3",
}
print("Required scripts:")
missing = []
for rel, phase in REQUIRED.items():
    ok = (WORK_DIR / rel).exists()
    print(f"  [{'ok' if ok else ' MISSING '}] {rel:<28} ({phase})")
    if not ok:
        missing.append(rel)

assert not missing, f"Missing {missing} -- is the clone at the right commit?"

run([PY, "scripts/env_check.py"])
run([PY, "-m", "src.data"])
run([PY, "scripts/check_data.py", "--quick"])

# Phase 2 gate. On CPU this takes ~50 min; on a T4 it is seconds, so it is
# cheap insurance to re-verify every model can still memorise 256 images
# right before committing 8 GPU-hours to training them.
run([PY, "scripts/model_checks.py"])

# Phase 3 gates, on the real GPU (spec 12.6: "smoke test passed on the GPU").
#  - smoke_test: every model through the real training loop, interrupted and
#    resumed, all output files checked. ~2-4 min on a T4.
#  - queue_selftest: the unattended queue itself -- child processes, deps,
#    LR auto-pick, a deliberately diverging run, relaunch/skip. Toy-sized runs
#    under results/_queue_test, so nothing it writes is mistaken for real work.
run([PY, "scripts/smoke_test.py"])
run([PY, "scripts/queue_selftest.py"])

print("\n>>> Compare the split sha256 above against your local run. They must match.")

In [ ]:
# =============================================================================
# 5. Budget planner (spec 12.4) -- times the real hardware, prints the queue
#    and a projected finish time, saves results/queue_9h.json.
#
# A committed run cannot wait for a "go", so the plan is applied as printed.
# Read it in the log afterwards; RUN_SUMMARY.md records what actually ran.
# =============================================================================
import os
os.chdir(WORK_DIR)
run([PY, "scripts/plan_budget.py", "--hours", str(BUDGET_HOURS)])

In [ ]:
# =============================================================================
# 6. THE LONG RUN -- LR sweep -> E1 -> E7/E5/E6 -> E2 -> E4 -> upgrades.
#
# Expect 6.5-8 h. Output streams live so the committed-run log stays useful,
# and is teed to results/run_all.log.
#
# Wrapped in try/except so that a failure still reaches the packaging cell
# below: partial results are worth far more than none.
# =============================================================================
import os, traceback
os.chdir(WORK_DIR)
(WORK_DIR / "results").mkdir(parents=True, exist_ok=True)

RUN_OK = False
try:
    run([PY, "scripts/run_all.py", "--mode", MODE],
        tee=str(WORK_DIR / "results" / "run_all.log"))
    RUN_OK = True
except Exception:
    print("\n" + "!" * 70)
    print("!! The queue exited non-zero. Packaging whatever completed.")
    print("!" * 70)
    traceback.print_exc()

In [ ]:
# =============================================================================
# 7. Package the results.
#
# Two archives:
#   results.zip        everything, checkpoints included (big)
#   results_light.zip  json/csv/md/png only -- this is what Phase 9 actually
#                      needs, and it survives Kaggle's output size cap
# =============================================================================
import os, shutil, zipfile
from pathlib import Path
os.chdir(WORK_DIR)

RESULTS = WORK_DIR / "results"
OUT = Path("/kaggle/working")

def human(n):
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024:
            return f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} TB"

if not RESULTS.exists() or not any(RESULTS.rglob("*")):
    print("results/ is empty -- nothing to package.")
else:
    # The smoke/self-test scratch dirs are not results.
    for scratch in ("smoke", "_queue_test", "queue_state"):
        shutil.rmtree(RESULTS / scratch, ignore_errors=True)
    full = shutil.make_archive(str(OUT / "results"), "zip", root_dir=str(RESULTS))
    print(f"results.zip        {human(Path(full).stat().st_size)}")

    light = OUT / "results_light.zip"
    # .npz = per-run test predictions (E7 agreement matrix, mistake gallery)
    keep = {".json", ".csv", ".md", ".png", ".yaml", ".log", ".txt", ".npz"}
    n = 0
    with zipfile.ZipFile(light, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(RESULTS.rglob("*")):
            if p.is_file() and p.suffix.lower() in keep:
                zf.write(p, p.relative_to(RESULTS))
                n += 1
    print(f"results_light.zip  {human(light.stat().st_size)}  ({n} files, no checkpoints)")

    total = sum(f.stat().st_size for f in OUT.rglob("*") if f.is_file())
    print(f"\ntotal /kaggle/working output: {human(total)}")

In [ ]:
# =============================================================================
# 8. Final summary.
# =============================================================================
from pathlib import Path
RESULTS = WORK_DIR / "results"

for name, n_lines in [("RUN_SUMMARY.md", None), ("progress.log", 40)]:
    p = RESULTS / name
    print("=" * 78)
    print(f"  {name}")
    print("=" * 78)
    if p.exists():
        text = p.read_text(encoding="utf-8", errors="replace")
        lines = text.splitlines()
        if n_lines and len(lines) > n_lines:
            print(f"... ({len(lines) - n_lines} earlier lines omitted)")
            lines = lines[-n_lines:]
        print("\n".join(lines))
    else:
        print(f"(not found -- {name} is written by scripts/run_all.py)")
    print()

fails = RESULTS / "failures"
if fails.exists():
    logs = sorted(fails.glob("*.log"))
    print(f"{len(logs)} failed run(s):")
    for f in logs:
        print(f"  - {f.name}")

---

## 📥 When you get back

1. **Download the archives** from the Output tab of the committed version:
   - `results_light.zip` — tables, figures, metrics, logs. This is what Phase 9 needs.
   - `results.zip` — the same plus checkpoints, if you want to re-run E5/E6/E7 locally.

2. **Unzip into the local repo** so the directory layout matches:
   ```powershell
   Expand-Archive -Path results_light.zip -DestinationPath .\results -Force
   ```

3. **Check three things before trusting the numbers:**
   - the split sha256 from cell 4 matches your local `check_data.py` output;
   - `results/RUN_SUMMARY.md` lists which upgrades actually ran and which runs failed;
   - no run in `results/failures/` is one you needed.

4. **Run Phase 9 locally** with the coding agent: figures, `results/REPORT.md`, the README, and the
   LinkedIn drafts.

### ⚠️ What the write-up must say (spec 12.7)

9h mode trades statistical power for wall-clock. The report has to state plainly:

- **1 seed for most results** → differences under ~1 percentage point between models are **not** meaningful.
- **100 epochs, not 200** → ViT and MLP-Mixer usually keep improving with longer training, so this
  setup slightly **favours the CNNs**. Say so rather than concluding "Transformers lose".
- **E2 points are step-matched but short** (7,040 steps), so absolute E2 accuracies sit well below E1's.
- **Which upgrades actually ran**, taken from `RUN_SUMMARY.md`.